In [17]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU 개수:", torch.cuda.device_count())
else:
    print("현재 PyTorch는 GPU를 사용할 수 없습니다.")

PyTorch: 2.14.1+cu126
CUDA 사용 가능: False
현재 PyTorch는 GPU를 사용할 수 없습니다.


In [19]:
import warnings
import os
from huggingface_hub import login

from dotenv import load_dotenv

warnings.simplefilter(action='ignore')
load_dotenv()
HF_TOKEN= os.getenv("HF_TOKEN")
login(token=HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


### 모델 정보 보기

| 모델 | 구조 | 대표 모델 | 사용 |
|--|--|--|--|
|Encoder-only | 인코더만 사용 | BERT | 입력 문장을 이해하고 분석하는 데 강함 |
|Decoder-only | 디코더만 사용 | GPT | 새로운 문장을 생성하는 데 강함 |
|Encoder + Decoder | 둘다 사용 | T5 | 둘 다 |


In [2]:
from huggingface_hub import HfApi

api = HfApi()
# bert : 인코더만 사용
info = api.model_info("klue/bert-base")

print("task : ", info.pipeline_tag)
print("다운로드 : ", info.downloads)
print("라이선스 : ", info.card_data.get('license'))

if info.safetensors:
    print("파라미터 수 : ", info.safetensors.total)


task :  fill-mask
다운로드 :  75750
라이선스 :  cc-by-sa-4.0
파라미터 수 :  111243522


In [3]:
for m in api.list_models(filter='text-classification',search='korean',sort='downloads',limit=10):
    print(f"{(m.downloads or 0):>10,} {m.id}")


     2,879 nlp04/korean_sentiment_analysis_kcelectra
     2,202 cringepnh/koelectra-korean-sentiment
       801 matthewburke/korean_sentiment
       536 WhitePeak/bert-base-cased-Korean-sentiment
        92 Seonghaa/korean-emotion-classifier-roberta
        78 hyun5ooo/deberta-v3-base-korean-ai-detect
        77 beomi/korean-hatespeech-multilabel
        48 GGARA02/kcelectra-korean-emotion
        47 beomi/korean-hatespeech-classifier
        44 2tle/korean-curse-detection


### 파이프라인
`pipeline('태스크명',model='모델id')`
device_map="auto" = gpu 있으면 사용

#### 1)감정분석

In [4]:
from transformers import pipeline

clf = pipeline('sentiment-analysis',model='distilbert/distilbert-base-uncased-finetuned-sst-2-english',device_map="auto",dtype=torch.float16)
# device_map="auto"
print(clf(['I love this phone.','The battery dies too fast.']))
print(clf(['배송이 너무 늦어요.']))

c:\source\llm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 423.60it/s]


[{'label': 'POSITIVE', 'score': 0.9998148083686829}, {'label': 'NEGATIVE', 'score': 0.9996964931488037}]
[{'label': 'POSITIVE', 'score': 0.561147153377533}]


#### 2)빈칸 채우기 - 한국어 BERT


In [10]:
import torch

clf = pipeline('fill-mask',model='klue/bert-base',device_map="auto",dtype=torch.float16)

#for r in clf("대한민국의 수도는 [MASK]이다."):
for r in clf("오늘의 점심은 [MASK]이다."):
    print(r['token_str'],round(r['score'], 3))

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 538.61it/s]
[transformers] BertForMaskedLM LOAD REPORT from: klue/bert-base
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


치킨 0.06
삼겹살 0.042
비빔밥 0.04
짜장면 0.024
도시락 0.023


#### 3) 제로샷
- 학습할 때 없었던 label 설정 가능

In [14]:
from transformers import pipeline
classifier = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",device_map="auto",dtype=torch.float16)

sequence_to_classify = "Angela Merkel ist eine Politikerin in Deutschland und Vorsitzende der CDU"
candidate_labels = ["politics", "economy", "entertainment", "environment"]
output = classifier(sequence_to_classify, candidate_labels, multi_label=False)
#print(output)
print(list(zip(output['labels'],[round(s, 3) for s in output['scores']])))


Loading weights: 100%|██████████| 202/202 [00:00<00:00, 1632.68it/s]


[('politics', 0.965), ('economy', 0.023), ('environment', 0.008), ('entertainment', 0.004)]


In [15]:
sequence_to_classify = "손흥민이 시즌 10호 골을 터뜨렸다."
candidate_labels = ["경제", "스포츠", "연예", "정치"]
output = classifier(sequence_to_classify, candidate_labels, multi_label=False)
#print(output)
print(list(zip(output['labels'],[round(s, 3) for s in output['scores']])))

[('스포츠', 0.497), ('연예', 0.293), ('경제', 0.108), ('정치', 0.102)]


#### ner(named entity recognition)
- 개체명 인식

In [23]:
ner = pipeline("ner", model="dslim/bert-base-NER",aggregation_strategy='simple',device_map="auto",dtype=torch.float16)
#example = "My name is Wolfgang and I live in Berlin"
example = "Elon Musk visited Seoul to meet Samsung executives."

for e in ner(example):
    print(e['entity_group'],e['word'],round(e['score'],3))


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 540.43it/s]
[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ORG Elon 0.806
PER Mu 0.51
ORG ##sk 0.683
LOC Seoul 1.0
ORG Samsung 0.999


#### vision-언어모델

In [ ]:
vision = pipeline("image-text-to-text", model="HuggingFaceTB/SmolVLM-256M-Instruct",device_map="auto",dtype=torch.float16)
#url = "https://png.pngtree.com/png-vector/20260522/ourmid/pngtree-a-white-sheep-lying-down-on-plain-background-png-image_19364753.webp"
url = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTv9GSy54opIzSS3kJHZtLDMO3Koh9OkIeAm5A6A9VBYQ&s=10"
messages = [
    {"role": "user", "content": [
        {'type':'image','url':url},
        {'type':'text','text':'Describe this image in one sentence'}
        ]}
    ]

print(vision(text=messages, max_new_tokens=40,return_full_text=False)[0]['generated_text'])


Loading weights: 100%|██████████| 471/471 [00:00<00:00, 1120.13it/s]


 In this image we can see a box with a shadow.


In [29]:
gen =  pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct",device_map="auto",dtype=torch.float16)
messages = [
    {"role": "user", "content":'HuggingFace를 한 문장으로 설명해줘'}
    ]

print(gen(messages, max_new_tokens=80)[0]['generated_text'][-1]['content'])

Loading weights: 100%|██████████| 338/338 [00:05<00:00, 66.10it/s]
Some parameters are on the meta device because they were offloaded to the cpu and disk.


[{'role': 'user', 'content': 'HuggingFace를 최대 두 문장으로 설명해줘'}, {'role': 'assistant', 'content': 'Hugging Face는 전문가들의 모임에서 만들어진 커뮤니티로, 자연어 처리(NLP)에 관련된 연구와 코드를 공유하는 플랫폼입니다. 이 플랫폼은 다양한 NLP 모델을 제공하며, 이를 통해 학습하고 사용할 수 있습니다. Hugging Face의 가장 주요 기능 중 하나는'}]
